In [ ]:
pip install --upgrade tiktoken

In [ ]:
import tiktoken

def count_tokens(text, model="gpt-5.2"):
    try:
        # Try to get the encoding specific to the model name
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # Fallback to o200k_base (standard for GPT-4o/GPT-5 series)
        print("Warning: Model not found. Falling back to o200k_base.")
        encoding = tiktoken.get_encoding("o200k_base")

    tokens = encoding.encode(text)
    return len(tokens)

my_text = "Here is my input for the new model."
print(f"Token count: {count_tokens(my_text)}")

In [ ]:
#!/usr/bin/env python3
import argparse
import os
import subprocess
import sys
import time
import json
import requests
import statistics
import concurrent.futures
from collections import deque
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple


# ---------- Config ----------

IGNORED_DIRS = {
    ".git",
    "node_modules",
    "__pycache__",
    ".mypy_cache",
    ".pytest_cache",
    ".venv",
    "venv",
    "dist",
    "build",
    ".idea",
    ".vscode",
    "css",
    "include",
    "vendor",
    "vendors",
    "libs",
    "lib",
    "third_party",
    "third-party",
    "site-packages",
    ".next",
    ".nuxt",
    ".angular",
    ".cache",
    "public",
    "static",
    "target",
    ".gradle",
    ".husky",
    ".pnpm",
    "jspm_packages",
    "bower_components",
    "coverage",
    "htmlcov",
}

# Known file names to ignore (lock files, generated deps, etc.)
IGNORED_FILE_NAMES = {
    "package-lock.json",
    "pnpm-lock.yaml",
    "yarn.lock",
    "poetry.lock",
    "Pipfile.lock",
    "Cargo.lock",
    "composer.lock",
    "Gemfile.lock",
    "go.sum",
    "go.work.sum",
    ".gitignore",
}

# Extensions to ignore completely (lock-like)
IGNORED_FILE_EXTS = {
    ".lock",
}

# Obvious binary / multimedia / big artifacts we don't care about
BINARY_EXTS = {
    # images
    ".png", ".jpg", ".jpeg", ".gif", ".bmp", ".webp", ".svg", ".ico",
    # audio
    ".mp3", ".wav", ".ogg", ".flac", ".m4a",
    # video
    ".mp4", ".mkv", ".avi", ".mov", ".webm",
    # archives & binaries
    ".zip", ".tar", ".gz", ".tgz", ".bz2", ".7z", ".rar",
    ".exe", ".dll", ".so", ".dylib",
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
}

# Extensions we treat as "likely text" even without probing
PREF_TEXT_EXTS = {
    ".py", ".go", ".js", ".jsx", ".ts", ".tsx", ".java", ".cs", ".php", ".rb",
    ".rs", ".c", ".cc", ".cpp", ".h", ".hpp",
    ".lua", ".sh", ".bash", ".zsh", ".ps1",
    ".tf", ".tfvars",
    ".yaml", ".yml", ".json", ".toml", ".ini", ".cfg",
    ".md", ".rst", ".txt",
    ".html", ".htm", ".css", ".scss", ".less",
    ".svelte", ".vue",
}

DEFAULT_GEMINI_MODEL = "gemini-3-pro-preview"
GEMINI_URL = "https://generativelanguage.googleapis.com/v1beta/models/{model_id}:generateContent"

DEFAULT_CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
CLAUDE_URL = "https://api.anthropic.com/v1/messages"

DEFAULT_OPENAI_MODEL = "gpt-5.2-2025-12-11"
OPENAI_URL = "https://api.openai.com/v1/chat/completions"


# ---------- Repo / FS helpers ----------

def run_git_clone(repo_url: str, dest_dir: Path) -> None:
    if dest_dir.exists():
        print(f"[INFO] Destination {dest_dir} already exists, skipping clone.", file=sys.stderr)
        return
    print(f"[INFO] Cloning {repo_url} into {dest_dir} ...", file=sys.stderr)
    subprocess.run(["git", "clone", repo_url, str(dest_dir)], check=True)


def should_ignore_file(path: Path) -> bool:
    name = path.name
    ext = path.suffix.lower()

    if name in IGNORED_FILE_NAMES:
        return True
    if ext in IGNORED_FILE_EXTS:
        return True
    # ignore obvious minified bundles
    if name.endswith(".min.js") or name.endswith(".min.css"):
        return True

    return False


def is_probably_text_file(path: Path, max_sample_bytes: int = 2048) -> bool:
    """
    Heuristic to skip obvious binaries; prefer text-like extensions,
    otherwise try to decode a small chunk as UTF-8.
    """
    if should_ignore_file(path):
        return False

    ext = path.suffix.lower()

    if ext in BINARY_EXTS:
        return False

    if ext in PREF_TEXT_EXTS:
        return True

    try:
        with path.open("rb") as f:
            chunk = f.read(max_sample_bytes)
        if not chunk:
            return True
        if b"\x00" in chunk:
            return False
        try:
            chunk.decode("utf-8")
            return True
        except UnicodeDecodeError:
            return False
    except OSError:
        return False


def read_file_text(path: Path) -> str:
    try:
        with path.open("r", encoding="utf-8", errors="replace") as f:
            return f.read()
    except OSError as e:
        print(f"[WARN] Failed to read {path}: {e}", file=sys.stderr)
        return ""


def bfs_directories(root: Path) -> Iterable[Path]:
    """
    Breadth-first traversal over directories starting at root.
    """
    queue = deque([root])
    while queue:
        current = queue.popleft()
        yield current

        try:
            entries = sorted(current.iterdir(), key=lambda p: p.name)
        except OSError as e:
            print(f"[WARN] Cannot list {current}: {e}", file=sys.stderr)
            continue

        for entry in entries:
            if entry.is_dir():
                if entry.name in IGNORED_DIRS:
                    continue
                queue.append(entry)


def collect_files_in_dir(directory: Path) -> List[Path]:
    try:
        entries = sorted(directory.iterdir(), key=lambda p: p.name)
    except OSError as e:
        print(f"[WARN] Cannot list files in {directory}: {e}", file=sys.stderr)
        return []
    return [p for p in entries if p.is_file()]


# ---------- Build repo context for evaluation ----------

def build_repo_files_block(
    root: Path,
    max_file_size_mb: float,
    max_file_chars: int,
    max_total_chars: int,
    allowed_files: List[str] = []
) -> Tuple[str, List[str]]:
    """
    Build a single big <FILES>...</FILES> section containing as many files
    as possible without exceeding max_total_chars.

    - Skips files > max_file_size_mb (in bytes).
    - Skips files whose text length > max_file_chars.
    - Stops adding more files when combined length exceeds max_total_chars.
    """
    max_file_size_bytes = int(max_file_size_mb * 1024 * 1024)

    allowed_files = set(allowed_files)
    added_allowed_files = 0

    total_chars = 0
    included_paths: List[str] = []
    file_blocks: List[str] = []

    print(f"[INFO] Building repo context from {root}", file=sys.stderr)

    for directory in bfs_directories(root):
        files = collect_files_in_dir(directory)
        for path in files:
            #print(path.relative_to(root).as_posix(), path.relative_to(root).as_posix(), path.relative_to(root).as_posix() in allowed_files)
            if allowed_files and path.relative_to(root).as_posix() not in allowed_files:
              continue
            if should_ignore_file(path):
                continue
            try:
                size = path.stat().st_size
            except OSError:
                continue

            if size > max_file_size_bytes:
                print(f"[INFO] Skipping big file {path} ({size} bytes)", file=sys.stderr)
                continue

            if not is_probably_text_file(path):
                continue

            text = read_file_text(path)
            if not text.strip():
                continue

            char_count = len(text)
            if char_count > max_file_chars:
                print(
                    f"[INFO] Skipping long file {path} with {char_count} chars (> max_file_chars={max_file_chars})",
                    file=sys.stderr,
                )
                continue

            if allowed_files:
              print(f"[WARN] {len(allowed_files)} allowed files specified and {path} selected")
              added_allowed_files += 1

            block = f'<FILE path="{path.relative_to(root).as_posix()}">\n{text}\n</FILE>\n\n'
            block_len = len(block)

            if total_chars + block_len > max_total_chars:
                print(
                    f"[INFO] Reached max_total_chars={max_total_chars}, stopping file collection.",
                    file=sys.stderr,
                )
                return "<FILES>\n" + "".join(file_blocks) + "</FILES>", included_paths

            file_blocks.append(block)
            included_paths.append(path.relative_to(root).as_posix())
            total_chars += block_len

    if allowed_files:
      print(f"[INFO] Added {added_allowed_files}/{len(allowed_files)} allowed files")

    return "<FILES>\n" + "".join(file_blocks) + "</FILES>", included_paths


# ---------- Evaluation prompt ----------

def build_eval_instructions() -> str:
    return """You are an expert software architect and static analysis judge.

You will receive:
- The source code of a repository (as a set of <FILE> blocks inside <FILES>...</FILES>).
- A system schema that claims to describe this repository (inside <SCHEMA>...</SCHEMA>).

Your task is to evaluate how well the system schema matches the provided codebase content.

System schema is supposed to describe both static and dynamic aspects of the system, including:
- system structure and services,
- runtime/deployment configuration (runtime, ci, dependencies),
- APIs and data models,
- internal components (databases, caches, structs, tables, methods, jobs/queues/topics, etc.),
- relationships and annotations (@PATH, @CALLS, @BASE, @DELETED, etc.).

The system schema inside <SCHEMA> may be either:
  - SiMAL (DSL text) or a JSON object.
  - If <SCHEMA> contains a JSON object (starts with "{" or wrapped in ```json), treat it as JSON.
  - Regardless of schema type, evaluate the semantics against <FILES>. Do not penalize formatting differences inherent to the chosen schema type.

You MUST carefully read the code and the schema and then answer strictly in JSON.

============================================================
INPUT FORMAT

1. Repository source code:
<FILES> <FILE path="relative/path.ext"> ...file content... </FILE> ... </FILES>

2. System schema to evaluate:
<SCHEMA> ...System schema text... </SCHEMA>
============================================================
EVALUATION INSTRUCTIONS

A) Evidence-first judging (anti-hallucination)

  - Only use evidence that is present in the provided <FILES> content and the given <SCHEMA>.
  - Do NOT assume files exist if they are not shown in <FILES>.
  - Assume <FILES> provides a complete and representative view of the repository necessary for evaluation.
  - Therefore, absence of evidence in <FILES> counts as evidence of absence, unless the input explicitly says the context is partial/truncated.
  - Treat schema claims that are not supported by <FILES> as errors (counted), not “unverifiable”.

B) Applicability handling (required; prevents unfair penalties)

Before assigning each category score, classify the category as one of:
  - Applicable: the aspect clearly exists in the provided <FILES>.
  - Not Applicable (N/A): the repo genuinely does not have that aspect by design (e.g., it is a library/package with no API surface; no runtime/deploy manifests; no CI config).
  - A category may be N/A only if the schema also does NOT claim that aspect exists. Example: If schema contains any api/routes/RPCs/etc, then api_accuracy_score cannot be -1, even if code doesn’t show an API.
  - N/A requires evidence from <FILES>, e.g. "no server entrypoint, no router/controller files, no OpenAPI/proto, no framework imports".

Scoring rules for applicability:
  - If Not Applicable (N/A): assign score = -1 for that category, and add a note stating it is N/A (with brief evidence, e.g., "library/no server entrypoints found").
  - If Applicable: assign score normally (0–5) using the rubric below.

This rule applies to every category score:
  - schema_coverage_score
  - schema_accuracy_score
  - api_accuracy_score
  - structure_accuracy_score
  - annotation_quality_score
  - non_hallucination_score

C) Definitions: major vs minor components

  - Major components include any of:
    - top-level service/microservice/app/module boundary (including monolith main modules),
    - public API surface (HTTP routes/controllers, gRPC services, GraphQL schema/resolvers, event topics/handlers, etc.),
    - primary persistence layer (main database, migrations, primary tables),
    - primary cache/broker (Redis, Kafka/RabbitMQ/NATS, etc.) when used,
    - CI pipeline and deployment/runtime configs when clearly present (Dockerfile, docker-compose, k8s manifests, Helm charts, CI config),
    - core domain models/entities/classes/structs/methods used widely by the system.

  - Minor components include:
    - helpers/utilities, internal libraries, small config files,
    - secondary/auxiliary tables, caches, background jobs that are not central,
    - small DTOs or thin wrappers.

D) Comparison tasks
  1. Coverage
    - Are all major services/modules represented?
    - Are key components (e.g., databases, caches, main models, important jobs/tasks) covered?
    - Are important runtime/ci/dependencies captured when clearly present in code/configs?
    - Count missed components as:
      - missed_major_components: clearly present major items in <FILES> that are absent from <SCHEMA>.
      - missed_minor_components: clearly present minor items in <FILES> that are absent from <SCHEMA>.
  2. Accuracy
    - When something appears in the schema, does it actually exist in the provided code/configs?
    - Are service names, component names, descriptions, algos, fields, methods and relationships consistent with the code?

  3. APIs
    - Do API endpoints in the schema match routes/handlers/controllers defined in code/configs?
    - Consider HTTP, gRPC, GraphQL, event/topic-based, websocket-like endpoints if present.
    - Matching guidance:
      - Incorrect if: wrong HTTP method, wrong path, wrong handler/service name, endpoint not present at all, or wrong gRPC method/service name.
      - Acceptable abstraction if: schema omits some optional fields or simplifies types while preserving correct endpoint identity.
      - Count incorrect_api_signatures only when there is clear evidence the schema endpoint is wrong.

  4. Internal structure (data models and components)
    - For structs/classes/models/tables:
      - Field/method names should match approximately (e.g., do not penalize snake_case vs camelCase differences).
      - Field/method types can be approximate (e.g., string vs varchar) unless the code clearly specifies exact types.
      - Visibility markers (+/-/#) should match when inferable (language rules, exported/unexported in Go, public/private in TS/Java/etc).

    - Count:
      - incorrect_struct_or_table_definitions when a struct/table is clearly mismodeled (wrong key fields/methods, missing major fields/methods, or describes non-existent type as if it exists).
      - incorrect_visibility_flags only when visibility is inferable and the schema is clearly wrong.

  5. Annotations and relationships
    - @PATH quality:
      - @PATH annotation is used to define source file paths for components/services/APIs.
      - Prefer treating @PATH as valid if it points to an existing file/directory prefix among provided <FILES> paths. That file/directory should contain relevant code for the schema element.
      - If @PATH has no path or an obviously incorrect path (e.g., random string, wrong file extension), count it as wrong.

    - @CALLS / @BASE quality:
      - @CALLS annotation specifies which other services/components/APIs are invoked by this element.
      - @BASE annotation specifies inheritance or composition relationships. For instance, production deployment config might be based on development config (reusing some settings) and instead of duplicating everything, @BASE could be used to indicate inheritance.
      - Validate when inferable from imports, client usage, references, or config wiring in provided <FILES>.
      - In case of empty or obviously incorrect @CALLS/@BASE (e.g., random strings, wrong names), count as wrong.

    - Non-hallucination (spurious items)
      - spurious_components: components/services/APIs that are asserted in <SCHEMA> but clearly do NOT exist in provided <FILES>.

E) Schema alignment strictness: acceptable vs penalize

E1. General rule (evidence-backed claims):
  - Every specific claim in <SCHEMA> must be backed by evidence in <FILES>.
  - If a claim cannot be backed (in STRICT MODE), penalize it.

E2. Acceptable abstraction (DO NOT penalize)
These sample differences are allowed if identity matches:
  - Naming style only: user_id vs userId, SidebarMenuReducer vs sidebar_menu_reducer.
  - Type approximation: string vs VARCHAR, UUID vs string, int vs number (unless exact types are explicitly declared in code/migrations).
  - Coarser grouping: Schema groups many files into one “module/service” entry, if that grouping is consistent with folder boundaries/import structure.

E3. Must be treated as errors (penalize)
These are sample schema inaccuracies:
  - Wrong endpoint identity (count as incorrect API signature), e.g.:
    - wrong service/method/RPC/resolver/op name, wrong path string, wrong handler/controller mapping.
  - Wrong component identity (count as spurious or incorrect struct/table definition), e.g.:
    - Schema asserts a component/type/table that does not exist in <FILES>.
    - Schema assigns a component to the wrong file/module (bad @PATH) when the correct location is clear.
  - Wrong fields/methods where code is explicit (count as incorrect struct/table definition), e.g.:
    - Schema claims a field/method exists but it doesn’t.
    - Schema omits a field/method that is clearly part of the core type API.
  - Contradictory or unsupported descriptions (penalize even if “just text”), e.g.:
    - Schema says “uses JWT auth” but no JWT/auth middleware/config is present.
    - Schema says “stores sessions in Redis” but Redis isn’t used anywhere and another store is shown.
    - Schema claims “idempotent endpoint” but code clearly mutates state and no idempotency mechanism exists.
  - Relationship claims without evidence (penalize), e.g.:
    - @CALLS(X) when no wiring indicates it.
    - @BASE(A) when there is no inheritance/overlay.

E4. Precision requirement for descriptions

  - If <SCHEMA> uses strong verbs like "uses / implements / guarantees / enforces / authenticates / encrypts / retries / caches", treat as falsifiable claims that must be evidenced.
  - If description is purely high-level (“handles users”), do not penalize.

Examples: acceptable vs error
  1. Endpoints
    - OK: schema says GET /users/{id} while code is GET /users/:id (same identity)
    - ERROR: schema says POST /users/{id} but code uses GET /users/:id → incorrect_api_signatures += 1
    - ERROR: schema says endpoint exists but there is no matching route string or handler mapping in code → incorrect_api_signatures += 1

  2. Models / structs
    - OK: schema says email: string, code uses email: varchar(255)
    - ERROR: schema includes passwordHash but model has password_digest and there is no mapping/alias logic → incorrect_struct_or_table_definitions += 1
    - ERROR: schema omits a central field used everywhere (id, uuid) when model definition explicitly includes it → incorrect_struct_or_table_definitions += 1

  3. Descriptions
    - OK: "handles authentication" (vague)
    - ERROR: "uses JWT middleware" but no JWT library/config/middleware exists in <FILES> → missed_minor_components += 1
    - ERROR: "caches responses for 5m" but no cache headers/middleware/cache store usage exists → missed_minor_components += 1

  4. Relationships
    - OK: schema omits @CALLS even if code hints it (optional)
    - ERROR: schema asserts @CALLS(payment_service) but no client/import/config refers to it → spurious_components += 1

F) Deterministic penalty mapping:

- Any wrong endpoint method/path/service name → incorrect_api_signatures += 1.
- Any missing major endpoint group that exists in code → missed_major_components += 1.
- Any incorrect/extra/missing core field or method on a major model/component → incorrect_struct_or_table_definitions += 1.
- Any wrong relationship claim (@CALLS/@BASE) that is unsupported → spurious_components += 1.
- Any "strong verb" claim in description that is unsupported → missed_minor_components += 1.
- If any of the above occurs, relevant subscore must be ≤ 4.

G) Required evidence in notes
  - Every entry in notes.major_issues and notes.minor_issues MUST include:
    - at least one concrete reference to provided files (file path and, if possible, symbol/endpoint string), AND
    - the related schema element name (service/component/API) when applicable.
  - For spurious items, include both:
    - schema element name, AND
    - evidence that it is absent (e.g., searched relevant paths/keywords in provided <FILES> and found none).
  - When a category is N/A, record that in notes.major_issues with brief evidence.

H) Required evaluation procedure (coverage-first)

Before scoring, do this:
  1. Extract a ground-truth checklist from <FILES>:
    - services/modules (top-level dirs, packages, apps),
    - API surfaces (route strings, controller/resolver names, gRPC service defs),
    - primary models/components (exported types, DB tables/migrations, key reducers/stores),
    - runtime/deploy/CI files if present.
  2. Compare schema to checklist item-by-item.
  3. Anything in checklist missing from schema → count as missed.
  4. Anything in schema not found in checklist → count as spurious/incorrect.

And enforce:
  - notes.comparisons must include at least 3 checklist-vs-schema comparisons with concrete evidence (file path + symbol/string).
  - If overall_score >= 95, require at least 5 such comparisons.

I) Handling of external/third-party sources
  1. Absence of an artifact in <FILES> counts as evidence of absence only for repo-owned artifacts, such as:
    - source files/classes/structs/functions defined in the repo,
    - repo-defined routes/controllers/gRPC services,
    - repo-defined DB migrations/tables/schemas,
    - repo-defined test/CI/deploy/runtime configs.
    - repo-defined examples, readme docs, or other artifacts.

  2. Third-party / external artifacts are exempt from "absence = absence". The schema is allowed to mention external dependencies/tools/frameworks that are not fully present in <FILES> (because they live outside the repository), but the judge must apply these rules:
    1) What counts as “externally sourced” content
      Treat a schema item as external if it refers to:
      - a known third-party library/framework/tool (e.g., React Router, Redux, Spring Boot, Gin, Echo, Fiber, NestJS, Kubernetes, Docker, Postgres),
      - a hosted service (AWS, GCP, Auth0, Stripe),
      - anything whose implementation is not expected to exist in the repo.
    2) Evidence required to accept external mentions (strict but fair)
       External mentions are acceptable and not penalized if either:
       - there is repo evidence of usage in <FILES> (e.g., import "react-router-dom", require("express"), go.mod, package.json, pom.xml, requirements.txt, Cargo.toml, Dockerfile, k8s manifests), or
       - the schema marks the statement as explicitly external/assumed, in which case it is not counted as wrong.
    3) What is still an error (even for third-party)
       Penalize the schema if it:
       - claims the repo implements a third-party component (as if it were repo-owned) when it clearly does not,
       - claims a specific external behavior/config as a fact without evidence (e.g., “JWT auth enforced” / “rate limit 10/min” / “cache TTL 5m”) and does not mark it as assumed,
       - invents external dependencies with no evidence of usage and not marked external/assumed.

  3. Suggested counting:
    - Unsupported external dependency mention (not evidenced, not marked assumed) → spurious_components += 1 (minor unless it drives architecture)
    - Unsupported strong behavioral claim about third-party config → missed_minor_components += 1

  4. Scoring impact of external-only information
    - External/assumed claims must not increase subscores.
    - Only repo-evidenced items should contribute to "excellent" coverage/accuracy.

============================================================
SCORING

All scores must be integers within the specified ranges:
  - schema_coverage_score: 0–5
  - schema_accuracy_score: 0–5
  - api_accuracy_score: 0–5
  - structure_accuracy_score: 0–5
  - annotation_quality_score: 0–5
  - non_hallucination_score: 0–5
  - overall_score: 0–100

0–5 rubric (apply consistently):
  - 5 = Excellent: near-complete and correct; only tiny issues.
  - 4 = Good: mostly correct; a few minor issues or small omissions.
  - 3 = Fair: partial; notable omissions or several mistakes but still usable.
  - 2 = Poor: many misses and/or several wrong assertions; hard to use.
  - 1 = Very poor: mostly incorrect; minimal alignment with code.
  - 0 = Unusable: does not match code at all or cannot be evaluated.
  - -1 = Not Applicable (N/A): aspect does not exist in the repo by design.

Verdict MUST be derived from overall_score using these thresholds:
  - 90–100: "excellent"
  - 75–89: "good"
  - 55–74: "fair"
  - 30–54: "poor"
  - 0–29: "unusable"

Overall score MUST be computed deterministically as follows:
  1. Compute a weighted base from subscores (each 0–5):
    - schema_coverage_score weight: 20
    - schema_accuracy_score weight: 20
    - api_accuracy_score weight: 20
    - structure_accuracy_score weight: 15
    - annotation_quality_score weight: 10
    - non_hallucination_score weight: 15

    Let weighted_base = round_half_up(
      (schema_coverage_score/5)*20 +
      (schema_accuracy_score/5)*20 +
      (api_accuracy_score/5)*20 +
      (structure_accuracy_score/5)*15 +
      (annotation_quality_score/5)*10 +
      (non_hallucination_score/5)*15
    )

    If any subscore is -1 (N/A), exclude it from both numerator and denominator when computing weighted_base.

    Where round_half_up(x) rounds to the nearest integer, with .5 rounded up.

  2. Apply penalties from counts (cap final result to 0–100):

    penalty = 7 * missed_major_components
      + 2 * missed_minor_components
      + 4 * spurious_components
      + 3 * incorrect_api_signatures
      + 3 * incorrect_struct_or_table_definitions
      + 1 * incorrect_visibility_flags

overall_score = clamp(weighted_base - penalty, min=0, max=100)
Where clamp(x, 0, 100) = min(max(x, 0), 100).

Counts MUST be non-negative integers:
  - missed_major_components
  - missed_minor_components
  - spurious_components
  - incorrect_api_signatures
  - incorrect_struct_or_table_definitions
  - incorrect_visibility_flags

NOTE ON SCORING CALCULATION:
  - You must output the raw sub-scores (0-5) or -1 and the raw counts of errors.
  - You should attempt to calculate the overall_score yourself to determine the verdict, but ensure your counts and sub-scores justify that verdict.
  - If a category is N/A, do not create missed counts for that category (missed_major/minor), but you may still count spurious/incorrect items if the schema asserts them.

============================================================
OUTPUT FORMAT (STRICT)

You MUST output a single JSON object and nothing else.
The JSON must have exactly the following top-level keys (no more, no less):

{
    "overall_score": int,
    "verdict": "excellent" | "good" | "fair" | "poor" | "unusable",
    "scores": {
        "schema_coverage_score": int (0–5 or -1 for N/A),
        "schema_accuracy_score": int (0–5 or -1 for N/A),
        "api_accuracy_score": int (0–5 or -1 for N/A),
        "structure_accuracy_score": int (0–5 or -1 for N/A),
        "annotation_quality_score": int (0–5 or -1 for N/A),
        "non_hallucination_score": int (0–5 or -1 for N/A)
    },
    "counts": {
        "missed_major_components": int,
        "missed_minor_components": int,
        "spurious_components": int,
        "incorrect_api_signatures": int,
        "incorrect_struct_or_table_definitions": int,
        "incorrect_visibility_flags": int
    },
    "notes": {
        "major_issues": [string],
        "minor_issues": [string],
        "comparisons": [string],
        "suggested_improvements": [string]
    }
}

Additional output constraints:
  - Do NOT include any comments, markdown, or additional keys.
  - Do NOT wrap the JSON in backticks.
  - Ensure scores are in range and counts are non-negative integers.
  - Ensure verdict matches overall_score thresholds exactly.
  - notes arrays may be empty but must be present.
  - Scores may be -1 only to indicate N/A; otherwise must be 0–5.

Now read the inputs and produce the final evaluation JSON.""".strip()


def build_eval_prompt(
    schema_text: str,
    files_block: str,
) -> str:
    instructions = build_eval_instructions()
    schema_block = f"<SCHEMA>\n{schema_text}\n</SCHEMA>"
    # files_block already wrapped in <FILES>...</FILES>
    return f"{instructions}\n\n{files_block}\n\n{schema_block}"


# ---------- Gemini call (LLM-as-a-Judge) ----------

def call_gemini_judge(
    model: str,
    api_key: str,
    prompt: str,
    log_input_path: Optional[str] = None,
    temperature: float = 0.1,
    max_retries: int = 3,
    retry_backoff_sec: float = 10.0,
) -> Dict:
    """
    Call Gemini (Generative Language API) to judge the schema.
    Expects JSON response with candidates[0].content.parts[0].text containing JSON.
    """
    if log_input_path:
        try:
            with open(log_input_path + f"_{str(time.time())}", "w", encoding="utf-8") as fw:
                json.dump({"model": model, "prompt": prompt}, fw, ensure_ascii=False, indent=2)
        except OSError as e:
            print(f"[WARN] Failed to log Gemini input to {log_input_path}: {e}", file=sys.stderr)

    params = {"key": api_key}
    body = {
        "model": model,
        "contents": [
            {
                "role": "user",
                "parts": [{"text": prompt}],
            }
        ],
        "generationConfig": {
            "temperature": float(temperature),
            "thinkingConfig": {
              "thinkingLevel": "HIGH",
            },
        },
    }

    last_error: Optional[Exception] = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                GEMINI_URL.format(model_id=model),
                params=params,
                data=json.dumps(body),
                headers={"Content-Type": "application/json"},
                timeout=300 * 3,
            )
            response.raise_for_status()
            data = response.json()

            try:
                text = data["candidates"][0]["content"]["parts"][0]["text"]
            except (KeyError, IndexError, TypeError) as e:
                raise RuntimeError(f"Unexpected Gemini response format: {e}, data={data}") from e

            # The model should return JSON as text. Parse it here.
            try:
                return json.loads(text)
            except json.JSONDecodeError as e:
                raise RuntimeError(f"Failed to parse model JSON output: {e}\nRaw text: {text}") from e

        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            last_error = e
            print(
                f"[WARN] Gemini transient error on attempt {attempt}/{max_retries}: {e}",
                file=sys.stderr,
            )
            if attempt < max_retries:
                time.sleep(retry_backoff_sec * attempt)
            continue

        except requests.exceptions.HTTPError as e:
            last_error = e
            status = e.response.status_code
            # Retry on Server Errors (5xx) or Rate Limits (429)
            if status == 429 or status >= 500:
                print(f"[WARN] Gemini API error {status} (attempt {attempt}/{max_retries})", file=sys.stderr)
                time.sleep(retry_backoff_sec * attempt)
            else:
                # Client error (400, 401, 403) -> Fail immediately
                print(f"[ERROR] Gemini client error {status}: {e}", file=sys.stderr)
                break

        except Exception as e:
            last_error = e
            print(f"[ERROR] Gemini unexpected error: {e}", file=sys.stderr)
            break

    raise RuntimeError(f"Gemini judge call failed after {max_retries} attempts: {last_error}")

# ---------- Claude call (LLM-as-a-Judge) ----------

def call_claude_judge(
    model: str,
    api_key: str,
    prompt: str,
    temperature: float,
    log_input_path: Optional[str] = None,
    max_retries: int = 3,
    retry_backoff_sec: float = 5.0,
) -> Dict:
    """
    Call Claude with robust error handling, retries, and logging.
    Saves full response (including thinking) to a separate JSON file for every attempt.
    """
    # 1. Log input prompt if requested (once per function call)
    if log_input_path:
        try:
            with open(log_input_path + f"_{str(time.time())}", "w", encoding="utf-8") as fw:
                json.dump({"model": model, "prompt": prompt}, fw, ensure_ascii=False, indent=2)
        except OSError as e:
            print(f"[WARN] Failed to log Claude input: {e}", file=sys.stderr)

    headers = {
        "x-api-key": api_key,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
        # Enable 1M Context
        "anthropic-beta": "context-1m-2025-08-07"
    }

    body = {
        "model": model,
        "max_tokens": 64000,
        "temperature": 1,
        "thinking": {
            "type": "enabled",
            "budget_tokens": 32000
        },
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                CLAUDE_URL,
                headers=headers,
                data=json.dumps(body),
                timeout=300 * 3
            )
            response.raise_for_status()

            data = response.json()

            # --- Log the full raw output (with thinking) ---
            if log_input_path:
                try:
                    p = Path(log_input_path)
                    # Create a unique filename for this attempt:
                    # e.g., "logs/eval_claude.json" -> "logs/eval_claude_response_attempt_1.json"
                    response_log_path = p.parent / f"{p.stem}_response_attempt_{str(time.time())}.json"

                    with open(response_log_path, "w", encoding="utf-8") as f_log:
                        json.dump(data, f_log, ensure_ascii=False, indent=2)
                except Exception as log_err:
                    print(f"[WARN] Failed to save raw Claude output: {log_err}", file=sys.stderr)
            # ----------------------------------------------------

            # Find the actual text block (skipping the thinking block)
            text = ""
            found_text = False

            for block in data.get("content", []):
                if block.get("type") == "text":
                    text = block.get("text", "")
                    found_text = True
                    break

            if not found_text:
                # Dump structure for debugging if text is missing
                content_types = [b.get('type') for b in data.get('content', [])]
                raise ValueError(f"No text block found. Content types present: {content_types}")

            # Clean Markdown wrappers
            if "```json" in text:
                text = text.split("```json")[1].split("```")[0]
            elif "```" in text:
                text = text.split("```")[1].split("```")[0]

            return json.loads(text)

        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            last_error = e
            print(f"[WARN] Claude connection error (attempt {attempt}/{max_retries}): {e}")
            time.sleep(retry_backoff_sec * attempt)

        except requests.exceptions.HTTPError as e:
            last_error = e
            status = e.response.status_code

            # --- Read the error body ---
            try:
                error_body = e.response.json() # Try to parse JSON error
            except:
                error_body = e.response.text   # Fallback to raw text

            # Retry on Server Errors (5xx) or Rate Limits (429)
            if status == 429 or status >= 500:
                print(f"[WARN] Claude API error {status} (attempt {attempt}/{max_retries}): {error_body}", file=sys.stderr)
                time.sleep(retry_backoff_sec * attempt)
            else:
                # 400 Client Errors (Bad Request) - Fail immediately and show why
                print(f"[ERROR] Claude Client Error {status}: {json.dumps(error_body, indent=2)}", file=sys.stderr)
                break

        except Exception as e:
            print("Claude error:", e)
            last_error = e
            print(f"[ERROR] Claude unexpected error: {e}")
            break

    raise RuntimeError(f"Claude judge failed after {max_retries} attempts. Last error: {last_error}")

# ---------- GPT call (LLM-as-a-Judge) ----------

def call_openai_judge(
    model: str,
    api_key: str,
    prompt: str,
    temperature: float,
    log_input_path: Optional[str] = None,
    max_retries: int = 3,
    retry_backoff_sec: float = 5.0,
) -> Dict:
    """
    Call OpenAI API (Chat Completions) to judge the schema.
    Enforces JSON mode for reliability.
    """
    # 1. Log input prompt if requested
    if log_input_path:
        try:
            with open(log_input_path + f"_{str(time.time())}", "w", encoding="utf-8") as fw:
                json.dump({"model": model, "prompt": prompt}, fw, ensure_ascii=False, indent=2)
        except OSError as e:
            print(f"[WARN] Failed to log OpenAI input: {e}", file=sys.stderr)

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    body = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        #"temperature": temperature,
        "max_completion_tokens": 128000,
        "reasoning_effort": "high",
        "response_format": {"type": "json_object"}  # Force JSON output
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                OPENAI_URL,
                headers=headers,
                data=json.dumps(body),
                timeout=300
            )

            # --- Log raw response for debugging ---
            if log_input_path:
                try:
                    p = Path(log_input_path)
                    response_log_path = p.parent / f"{p.stem}_response_attempt_{str(time.time())}.json"
                    # We try to dump the full JSON, if it fails (non-json response), we dump text
                    try:
                        resp_data = response.json()
                        with open(response_log_path, "w", encoding="utf-8") as f_log:
                            json.dump(resp_data, f_log, ensure_ascii=False, indent=2)
                    except:
                        with open(response_log_path + ".txt", "w", encoding="utf-8") as f_log:
                            f_log.write(response.text)
                except Exception as log_err:
                    print(f"[WARN] Failed to save raw OpenAI output: {log_err}", file=sys.stderr)
            # --------------------------------------

            response.raise_for_status()
            data = response.json()

            content = data["choices"][0]["message"]["content"]

            # Clean markdown wrappers if present (even in JSON mode they sometimes appear)
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]

            return json.loads(content)

        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            last_error = e
            print(f"[WARN] OpenAI connection error (attempt {attempt}/{max_retries}): {e}")
            time.sleep(retry_backoff_sec * attempt)

        except requests.exceptions.HTTPError as e:
            last_error = e
            status = e.response.status_code
            try:
                err_body = e.response.json()
            except:
                err_body = e.response.text

            if status == 429 or status >= 500:
                print(f"[WARN] OpenAI API error {status} (attempt {attempt}/{max_retries}): {err_body}", file=sys.stderr)
                time.sleep(retry_backoff_sec * attempt)
            else:
                print(f"[ERROR] OpenAI Client Error {status}: {err_body}", file=sys.stderr)
                break

        except Exception as e:
            last_error = e
            print(f"[ERROR] OpenAI unexpected error: {e}", file=sys.stderr)
            break

    raise RuntimeError(f"OpenAI judge failed after {max_retries} attempts. Last error: {last_error}")


# ---------- Main orchestration ----------
import math

def recalculate_final_score(eval_json: Dict) -> Dict:
    """
    Deterministically recalculates the overall_score and verdict.
    Handles '-1' (N/A) by excluding those weights and normalizing the remaining
    score to a 0-100 scale before applying penalties.
    """
    scores = eval_json.get("scores", {})
    counts = eval_json.get("counts", {})

    # 1. Weights definitions (Total = 100)
    weights = {
        "schema_coverage_score": 20,
        "schema_accuracy_score": 20,
        "api_accuracy_score": 20,
        "structure_accuracy_score": 15,
        "annotation_quality_score": 10,
        "non_hallucination_score": 15,
    }

    # 2. Penalty definitions
    penalty_weights = {
        "missed_major_components": 7,
        "missed_minor_components": 2,
        "spurious_components": 4,
        "incorrect_api_signatures": 3,
        "incorrect_struct_or_table_definitions": 3,
        "incorrect_visibility_flags": 1,
    }

    # 3. Calculate Weighted Base (with N/A normalization)
    active_weight_sum = 0.0
    earned_weight_sum = 0.0

    for key, max_weight in weights.items():
        raw_val = float(scores.get(key, 0))

        # If N/A (-1), skip this category completely
        if raw_val == -1:
            continue

        # Accumulate the max possible weight for this active category
        active_weight_sum += max_weight

        # Accumulate the earned score (normalized to weight)
        # Formula: (score / 5.0) * weight
        earned_weight_sum += (max(0, raw_val) / 5.0) * max_weight

    # Normalize to 100-point scale
    # If everything is N/A (edge case), default to 0 to avoid div/0
    if active_weight_sum > 0:
        normalized_base = (earned_weight_sum / active_weight_sum) * 100.0
    else:
        normalized_base = 0.0

    # Rounding logic: round_half_up
    weighted_base = int(normalized_base + 0.5)

    # 4. Calculate Penalty
    total_penalty = 0
    for key, p_val in penalty_weights.items():
        count_val = int(counts.get(key, 0))
        total_penalty += count_val * p_val

    # 5. Final Score Clamp
    raw_final = weighted_base - total_penalty
    overall_score = max(0, min(100, raw_final))

    # 6. Derive Verdict
    if overall_score >= 90:
        verdict = "excellent"
    elif overall_score >= 75:
        verdict = "good"
    elif overall_score >= 55:
        verdict = "fair"
    elif overall_score >= 30:
        verdict = "poor"
    else:
        verdict = "unusable"

    # 7. Update JSON
    eval_json["manual_overall_score"] = overall_score
    eval_json["manual_verdict"] = verdict

    eval_json["_calculation_debug"] = {
        "active_weight_sum": active_weight_sum,
        "earned_weight_sum": earned_weight_sum,
        "weighted_base": weighted_base,
        "total_penalty": total_penalty,
        "raw_final": raw_final
    }

    return eval_json

# ---------- Aggregation Logic ----------

def calculate_aggregate_stats(all_results: Dict[str, List[Dict]]) -> Dict:
    """
    Calculates averages for individual models and a total average.
    """
    summary = {
        "total": {"avg_score": 0.0, "count": 0},
        "models": {}
    }

    grand_total_score = 0
    grand_total_count = 0

    for model_name, runs in all_results.items():
        if not runs: continue

        scores = [r.get("manual_overall_score", 0) for r in runs]
        avg = statistics.mean(scores) if scores else 0

        summary["models"][model_name] = {
            "avg_score": round(avg, 2),
            "runs": len(scores),
            "scores": scores
        }

        grand_total_score += sum(scores)
        grand_total_count += len(scores)

    if grand_total_count > 0:
        summary["total"]["avg_score"] = round(grand_total_score / grand_total_count, 2)

    summary["total"]["count"] = grand_total_count
    return summary

def evaluate_schema_for_repo(
    repo_url: Optional[str],
    repo_path: Optional[str],
    schema_path: str,
    dest: str,
    gemini_model: str,
    max_file_size_mb: float,
    max_file_chars: int,
    max_total_chars: int,
    eval_output: str,
    log_input_path: Optional[str],
    temperature: float,
    allowed_files: List[str]
) -> None:
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        print("ERROR: GEMINI_API_KEY environment variable is not set.", file=sys.stderr)
        sys.exit(1)

    # Determine repo root
    if repo_path:
        root = Path(repo_path).resolve()
    else:
        if not repo_url:
            print("ERROR: Either --repo-path or --repo-url must be provided.", file=sys.stderr)
            sys.exit(1)
        dest_dir = Path(dest).resolve()
        run_git_clone(repo_url, dest_dir)
        root = dest_dir

    if not root.exists():
        print(f"ERROR: Repo path {root} does not exist.", file=sys.stderr)
        sys.exit(1)

    schema_file = Path(schema_path).resolve()
    if not schema_file.exists():
        print(f"ERROR: Schema file {schema_file} does not exist.", file=sys.stderr)
        sys.exit(1)

    print(f"[INFO] Reading schema from {schema_file}", file=sys.stderr)
    try:
        schema_text = schema_file.read_text(encoding="utf-8")
    except OSError as e:
        print(f"ERROR: Failed to read schema file: {e}", file=sys.stderr)
        sys.exit(1)

    files_block, included_paths = build_repo_files_block(
        root=root,
        max_file_size_mb=max_file_size_mb,
        max_file_chars=max_file_chars,
        max_total_chars=max_total_chars,
        allowed_files=allowed_files
    )

    print(
        f"[INFO] Included {len(included_paths)} files in evaluation context (char budget {max_total_chars}).",
        file=sys.stderr,
    )

    prompt = build_eval_prompt(schema_text=schema_text, files_block=files_block)

    input_tokens = count_tokens(prompt)
    print(f"Total input tokens: {input_tokens}")

    eval_result = call_gemini_judge(
        model=gemini_model,
        api_key=api_key,
        prompt=prompt,
        log_input_path=log_input_path,
        temperature=temperature,
    )

    print("[INFO] Recalculating score deterministically...", file=sys.stderr)
    final_result = recalculate_final_score(eval_result)

    output_path = Path(eval_output).resolve()
    try:
        with output_path.open("w", encoding="utf-8") as fw:
            json.dump(final_result, fw, ensure_ascii=False, indent=2)
    except OSError as e:
        print(f"ERROR: Failed to write evaluation output to {output_path}: {e}", file=sys.stderr)
        sys.exit(1)

    print(f"[INFO] Evaluation written to {output_path}", file=sys.stderr)
    print(f"[RESULT] Verdict: {final_result.get('manual_verdict')} (Score: {final_result.get('manual_overall_score')})", file=sys.stderr)


# ---------- Main Runner ----------
# ---------- batch runners (sequential per model) ----------

def run_sequential_batch(
    model_type: str,
    model_id: str,
    api_key: str,
    iterations: int,
    prompt: str,
    temperature: float,
    log_input_path: Optional[str] = None
) -> List[Dict]:
    """
    Runs N iterations sequentially for a single provider using the robust call functions.
    """
    results = []

    # Select function based on type
    if model_type == "gemini":
        call_func = call_gemini_judge
        # If we are running both in parallel, we might want unique log files,
        # but for now we append type to filename if a path is provided
        if log_input_path:
            p = Path(log_input_path)
            model_log_path = str(p.parent / f"{p.stem}_{model_type}{p.suffix}")
        else:
            model_log_path = None
    elif model_type == "claude":
        call_func = call_claude_judge
        if log_input_path:
            p = Path(log_input_path)
            model_log_path = str(p.parent / f"{p.stem}_{model_type}{p.suffix}")
        else:
            model_log_path = None
    elif model_type == "openai":
        call_func = call_openai_judge
        if log_input_path:
            p = Path(log_input_path)
            model_log_path = str(p.parent / f"{p.stem}_{model_type}{p.suffix}")
        else:
            model_log_path = None

    print(f"[START] Batch of {iterations} runs for {model_type.upper()} ({model_id})...", file=sys.stderr)

    for i in range(iterations):
        start_time = time.time()
        try:
            raw_json = call_func(
                model=model_id,
                api_key=api_key,
                prompt=prompt,
                temperature=temperature,
                log_input_path=model_log_path,
                max_retries=5,
                retry_backoff_sec=5.0
            )

            # Recalculate
            final_json = recalculate_final_score(raw_json)

            # Metadata
            final_json["_meta"] = {
                "model": model_id,
                "provider": model_type,
                "iteration": i + 1,
                "duration_sec": round(time.time() - start_time, 2)
            }

            results.append(final_json)
            print(f"[{model_type.upper()} {i+1}/{iterations}] Success. Score: {final_json['manual_overall_score']}", file=sys.stderr)

        except Exception as e:
            print(f"[{model_type.upper()} {i+1}/{iterations}] FAILED: {e}", file=sys.stderr)
            results.append({
                "error": str(e),
                "manual_overall_score": 0,
                "manual_verdict": "error",
                "_meta": {"iteration": i+1, "failed": True}
            })

    return results


def evaluate_schema_for_repo(
    repo_path: str,
    schema_path: str,
    gemini_model: str,
    claude_model: str,
    openai_model: str,
    max_file_size_mb: float,
    max_file_chars: int,
    max_total_chars: int,
    eval_output: str,
    temperature: float,
    log_input_path: Optional[str],
    iterations: Optional[int] = 3,
    allowed_files: Optional[List[str]] = []
) -> None:

    # 1. Configuration
    gemini_key = os.getenv("GEMINI_API_KEY")
    claude_key = os.getenv("CLAUDE_API_KEY")
    openai_key = os.getenv("OPENAI_API_KEY")

    if not gemini_key:
        print("ERROR: GEMINI_API_KEY environment variable is not set.", file=sys.stderr)
        sys.exit(1)
    if not claude_key:
        print("ERROR: CLAUDE_API_KEY environment variable is not set.", file=sys.stderr)
        sys.exit(1)
    if openai_model and not openai_key:
        print("ERROR: OPENAI_API_KEY is required when openai_model is specified.", file=sys.stderr)
        sys.exit(1)

    # 2. Prepare Context (Git/File IO)
    if repo_path:
        root = Path(repo_path).resolve()
    else:
        print("ERROR: --repo-path must be provided.", file=sys.stderr)
        sys.exit(1)

    if not root.exists():
        print(f"ERROR: Repo path {root} does not exist.", file=sys.stderr)
        sys.exit(1)

    schema_file = Path(schema_path).resolve()
    if not schema_file.exists():
        print(f"ERROR: Schema file {schema_file} does not exist.", file=sys.stderr)
        sys.exit(1)

    try:
        schema_text = schema_file.read_text(encoding="utf-8")
    except OSError as e:
        print(f"ERROR: Failed to read schema file: {e}", file=sys.stderr)
        sys.exit(1)


    if allowed_files:
      print(f"[WARN] Allowed files list specified with {len(allowed_files)} files")

    files_block, included_paths = build_repo_files_block(
        root=root,
        max_file_size_mb=max_file_size_mb,
        max_file_chars=max_file_chars,
        max_total_chars=max_total_chars,
        allowed_files=allowed_files
    )

    print(f"[INFO] Included {len(included_paths)} files. Tokenizing prompt...", file=sys.stderr)
    full_prompt = build_eval_prompt(schema_text=schema_text, files_block=files_block)

    # Log input if requested
    if log_input_path:
        with open(log_input_path, "w", encoding="utf-8") as fw:
            json.dump({"prompt": full_prompt}, fw, indent=2)

    # 3. Execution (parallel providers, serial requests)
    all_results = {"gemini": [], "claude": []}

    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = {}

        # Schedule Gemini batch
        if gemini_key and gemini_model:
            f = executor.submit(
                run_sequential_batch,
                "gemini", gemini_model, gemini_key, iterations, full_prompt, temperature,
                log_input_path
            )
            futures[f] = "gemini"

        # Schedule Claude batch
        if claude_key and claude_model:
            f = executor.submit(
                run_sequential_batch,
                "claude", claude_model, claude_key, iterations, full_prompt, temperature,
                log_input_path
            )
            futures[f] = "claude"

        # Schedule OpenAI batch
        if openai_key and openai_model:
            f = executor.submit(
                run_sequential_batch,
                "openai", openai_model, openai_key, iterations, full_prompt, temperature,
                log_input_path
            )
            futures[f] = "openai"

        for future in concurrent.futures.as_completed(futures):
            provider = futures[future]
            try:
                all_results[provider] = future.result()
            except Exception as e:
                print(f"[CRITICAL] {provider} batch failed: {e}", file=sys.stderr)

    # 4. Aggregation
    summary = {"total_avg": 0.0, "models": {}}
    grand_total_score = 0
    grand_total_count = 0

    for model_name, runs in all_results.items():
        if not runs: continue
        valid_scores = [r.get("manual_overall_score", 0) for r in runs if "error" not in r]
        avg = statistics.mean(valid_scores) if valid_scores else 0

        summary["models"][model_name] = {
            "avg_score": round(avg, 2),
            "runs_attempted": len(runs),
            "valid_runs": len(valid_scores)
        }
        grand_total_score += sum(valid_scores)
        grand_total_count += len(valid_scores)

    if grand_total_count > 0:
        summary["total_avg"] = round(grand_total_score / grand_total_count, 2)

    final_output = {
        "summary": summary,
        "runs": all_results
    }

    # 5. Output
    output_path = Path(eval_output).resolve()
    with output_path.open("w", encoding="utf-8") as fw:
        json.dump(final_output, fw, ensure_ascii=False, indent=2)

    print(f"[RESULT] Combined Score: {summary['total_avg']}", file=sys.stderr)
    print(f"[INFO] Full detailed results written to {output_path}", file=sys.stderr)

In [ ]:
os.environ["GEMINI_API_KEY"] = "<YOUR_GEMINI_KEY>"
os.environ["CLAUDE_API_KEY"] = "<YOUR_CLAUDE_KEY>"
os.environ["OPENAI_API_KEY"] = "<YOUR_OPENAI_KEY>"

In [ ]:
evaluate_schema_for_repo(
  repo_path="/content/simal/spring-food-delivery-microservices/repo",
  schema_path="/content/simal/spring-food-delivery-microservices/simal_schemas/schema_output_step=60.txt",
  gemini_model="gemini-3-pro-preview",
  claude_model="claude-sonnet-4-5-20250929",
  openai_model="gpt-5.2-2025-12-11",
  max_file_chars=250_000,
  max_file_size_mb=2.0,
  max_total_chars=4_000_000,
  eval_output="/content/simal/spring-food-delivery-microservices/spring-food-delivery-microservices_simal_schema_eval.json",
  log_input_path="/content/simal/spring-food-delivery-microservices/spring-food-delivery-microservices_autoeval_simal_log_input_file.log",
  temperature=0.1,
  iterations=3,
  #allowed_files=allowed_files
)